# Thai Overt & Mimed Speech EEG: Within-Subject Frequency Band & ICA Analysis

This notebook covers the exploratory within-subject experiments and signal analyses:
1. **Dataset Overview**: Summary of the 20-word Thai Overt and Mimed Speech EEG dataset (2 cohorts × 10 provisional 200-row participant slots, 32 channels, 512 Hz, 80,000 trials).
2. **5-Step Preprocessing & Power Spectral Density (PSD)**: Microvolt scale restoration, Common Average Reference (CAR), bandpass filtering, and Welch PSD across frequency bands.
3. **Within-Subject 4-Class `EEGNet` Across Frequency Bands**: Chronological within-block split (64% Train / 16% Validation / 20% Test) on words `[5, 6, 7, 8]` across `Delta`, `Theta`, `Alpha`, `Beta`, `Gamma`, and `Broadband`.
4. **Independent Component Analysis (`Picard ICA` + `MNE-ICLabel`)**: 25-component decomposition and automated rejection of muscle (`>30%`), ocular, cardiac, and low-brain (`<=40%`) components prior to `EEGNet` re-evaluation.

> **Methodological Note**: Dataset specifications follow [Mendeley Data](https://data.mendeley.com/datasets/yzts9t69r2/1) and personal communication with the dataset research team. Because trials for each word are stored in a contiguous 200-trial block (and word 5 vs. words 6–8 belong to different 5-word recording groups), high within-subject accuracy reflects within-block discriminability and does not establish isolated semantic neural decoding or complete EMG removal.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from mne.preprocessing import ICA
from mne_icalabel import label_components

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.data_loader import DEFAULT_DATASET_DIR, VOCABULARY_METADATA
from src.within_subject.eegnet_frequency_ica import (
    build_within_subject_splits,
    drop_bad_trials,
    load_word_epochs,
    train_and_evaluate_eegnet,
)
from src.within_subject.plot_frequency_accuracy import generate_frequency_accuracy_outputs

plt.style.use("default")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["font.sans-serif"] = ["Tahoma", "Leelawadee UI", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"MNE version   : {mne.__version__}")
print(f"PyTorch device: {device}")
print(f"Dataset path  : {DEFAULT_DATASET_DIR}")

## 1. Dataset Overview

In [ ]:
summary_rows = []
total_overt_trials = 0
total_mimed_trials = 0

for word_id, meta in VOCABULARY_METADATA.items():
    overt_path = DEFAULT_DATASET_DIR / "overt" / f"overt_data_word_{word_id:02d}.npz"
    mime_path = DEFAULT_DATASET_DIR / "mime" / f"mime_data_word_{word_id:02d}.npz"

    o_trials = int(np.load(overt_path)["original_shape"][0]) if overt_path.exists() else 0
    m_trials = int(np.load(mime_path)["original_shape"][0]) if mime_path.exists() else 0
    total_overt_trials += o_trials
    total_mimed_trials += m_trials

    summary_rows.append({
        "Word ID": f"word_{word_id:02d}",
        "Cohort": meta["cohort"],
        "Recording Group": f"Group {meta['group']}",
        "Thai Word": meta["thai"],
        "English Meaning": meta["english"],
        "Category": meta["category"],
        "Overt Trials": o_trials,
        "Mimed Trials": m_trials,
        "Total Trials": o_trials + m_trials,
    })

df_dataset = pd.DataFrame(summary_rows)
print(f"Overt Trials: {total_overt_trials:,} | Mimed Trials: {total_mimed_trials:,} | Total: {total_overt_trials + total_mimed_trials:,}")
display(df_dataset)

## 2. Power Spectral Density (PSD) Across Frequency Bands

In [ ]:
epochs_demo, sfreq = load_word_epochs(word_id=1, modality="overt", dataset_dir=DEFAULT_DATASET_DIR, apply_baseline=False)
spectrum = epochs_demo.compute_psd(method="welch", fmin=1.0, fmax=100.0, n_fft=512, verbose=False)
psds, freqs = spectrum.get_data(return_freqs=True)

psd_bands = {
    "Delta (1-4 Hz)": (1.0, 4.0),
    "Theta (4-8 Hz)": (4.0, 8.0),
    "Alpha (8-12 Hz)": (8.0, 12.0),
    "Beta (12-30 Hz)": (12.0, 30.0),
    "Gamma (30-100 Hz)": (30.0, 100.0),
}

mean_psd_all = psds.mean(axis=(0, 1))
band_powers = {
    band: float(np.mean(mean_psd_all[(freqs >= f_lo) & (freqs <= f_hi)]))
    for band, (f_lo, f_hi) in psd_bands.items()
}

df_bands = pd.DataFrame(list(band_powers.items()), columns=["Frequency Band", "Mean Power (V²/Hz)"])
display(df_bands)

plt.figure(figsize=(8.5, 4.0))
plt.bar(band_powers.keys(), band_powers.values(), color=["#1B4F72", "#2874A6", "#148F77", "#E67E22", "#C0392B"])
plt.ylabel("Power Spectral Density (V²/Hz)")
plt.title("Average Band Power Comparison (Welch PSD)")
plt.xticks(rotation=15)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 3. Within-Subject 4-Class `EEGNet` Evaluation

- **Target Classes**: Words `[5, 6, 7, 8]` (4 classes, chance = 25%)
- **Split**: Chronological per subject and word block (64% Train / 16% Validation / 20% Test)
- **Model**: Individual `EEGNet` trained for 100 epochs per subject across 10 subjects

In [ ]:
SUBJECT_IDS = list(range(10))
WORDS = [5, 6, 7, 8]
MODALITY = "overt"  # "overt" or "mime"
L_FREQ = 0.5
H_FREQ = 45.0
EPOCHS = 100
BATCH_SIZE = 32

all_word_epochs = {}
for w in WORDS:
    ep, sfreq = load_word_epochs(w, modality=MODALITY, dataset_dir=DEFAULT_DATASET_DIR, apply_baseline=True)
    ep.filter(l_freq=L_FREQ, h_freq=H_FREQ, method="iir", iir_params=dict(order=4, ftype="butter"), verbose=False)
    if L_FREQ < 50.0 < H_FREQ:
        ep.filter(l_freq=51.0, h_freq=49.0, method="iir", iir_params=dict(order=4, ftype="butter"), verbose=False)
    all_word_epochs[w] = ep

subject_data = build_within_subject_splits(all_word_epochs, words=WORDS, subject_ids=SUBJECT_IDS)
subject_accuracies = train_and_evaluate_eegnet(
    subject_data,
    n_classes=len(WORDS),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    device=device,
)

print(f"=== Within-Subject 4-Class EEGNet ({MODALITY.upper()}, {L_FREQ}-{H_FREQ} Hz) ===")
for s, acc in subject_accuracies.items():
    print(f"Subject {s:02d} -> Test Accuracy: {acc:.2f}%")
vals = list(subject_accuracies.values())
print("=" * 52)
print(f"Mean Within-Subject Accuracy: {np.mean(vals):.2f}% ± {np.std(vals, ddof=1):.2f}%")
print("=" * 52)

### Within-Subject Frequency-Band Accuracy Summary

In [ ]:
csv_path, png_path = generate_frequency_accuracy_outputs(PROJECT_ROOT / "results" / "within_subject")
df_freq_scores = pd.read_csv(csv_path)
df_summary = (
    df_freq_scores.groupby(["band", "mode"], sort=False)["accuracy_percent"]
    .agg(Mean="mean", SD=lambda x: x.std(ddof=1))
    .unstack("mode")
    .round(2)
)
display(df_summary)

## 4. Artifact Removal via Picard ICA & MNE-ICLabel

- Decomposes each subject's trials into `25 ICs` using `Picard Extended Infomax ICA`.
- Classifies components with `MNE-ICLabel` and rejects `eye blink`, `heart beat`, `line noise`, `channel noise`, `Muscle > 30%`, or `Brain <= 40%`.

In [ ]:
raw_word_epochs = {
    w: load_word_epochs(w, modality=MODALITY, dataset_dir=DEFAULT_DATASET_DIR, apply_baseline=False)[0]
    for w in WORDS
}

subject_icas = {}
subject_labels = {}
subject_scores = {}

print(f"=== Running Picard ICA + ICLabel ({len(SUBJECT_IDS)} Subjects) ===\n")

for s in SUBJECT_IDS:
    sub_eps = mne.concatenate_epochs([raw_word_epochs[w][s * 200 : (s + 1) * 200] for w in WORDS], verbose=False)
    sub_for_ica = sub_eps.copy().filter(
        1.0, 100.0, method="iir", iir_params=dict(order=4, ftype="butter"), verbose=False
    )
    if sub_for_ica.info["sfreq"] > 250:
        sub_for_ica.resample(250.0, verbose=False)

    ica = ICA(n_components=25, method="picard", fit_params=dict(ortho=False, extended=True), random_state=42)
    ica.fit(sub_for_ica, decim=2, verbose=False)

    ic_labels = label_components(sub_for_ica, ica, method="iclabel")
    labels = ic_labels["labels"]
    scores = ica.labels_scores_

    exclude_idx = [
        i
        for i, (lbl, sc) in enumerate(zip(labels, scores))
        if (lbl in ["eye blink", "heart beat", "line noise", "channel noise"]) or (sc[1] > 0.30) or (sc[0] <= 0.40)
    ]
    if len(exclude_idx) == len(labels):
        exclude_idx.remove(int(np.argmax(scores[:, 0])))

    ica.exclude = exclude_idx
    subject_icas[s] = ica
    subject_labels[s] = labels
    subject_scores[s] = scores

    print(
        f"Subject {s:02d} -> Excluded {len(exclude_idx):02d}/{len(labels)} ICs "
        f"(Retained {len(labels) - len(exclude_idx):02d} ICs) | Excluded: {[f'IC{i:02d}' for i in exclude_idx]}"
    )

In [ ]:
sample_subject = SUBJECT_IDS[0]
ica_sample = subject_icas[sample_subject]
labels_sample = subject_labels[sample_subject]
scores_sample = subject_scores[sample_subject]
muscle_idx = [
    i for i, (lbl, sc) in enumerate(zip(labels_sample, scores_sample[:, 1])) if lbl == "muscle" or sc > 0.30
]

fig_topo = ica_sample.plot_components(show=False)
plt.show()

brain_pct = scores_sample[:, 0] * 100.0
muscle_pct = scores_sample[:, 1] * 100.0

fig, ax = plt.subplots(figsize=(13, 4))
x = np.arange(len(labels_sample))
width = 0.36
ax.bar(x - width / 2, brain_pct, width, label="Brain (%)", color="#2ca02c", alpha=0.88)
ax.bar(x + width / 2, muscle_pct, width, label="Muscle (%)", color="#d62728", alpha=0.88)

for idx in muscle_idx:
    ax.axvspan(
        idx - 0.45,
        idx + 0.45,
        color="red",
        alpha=0.14,
        label="Muscle Candidate (>30%)" if idx == muscle_idx[0] else "",
    )

ax.set_xticks(x)
ax.set_xticklabels([f"IC{i:02d}" for i in x], rotation=45)
ax.set_ylabel("Probability (%)")
ax.set_title(f"ICLabel Component Breakdown (Subject {sample_subject:02d}): Brain % vs Muscle %")
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
subject_data_clean = {}

for s in SUBJECT_IDS:
    s_tr, s_v, s_te = [], [], []
    ica_s = subject_icas[s]
    for w in WORDS:
        sub_raw = raw_word_epochs[w][s * 200 : (s + 1) * 200].copy()
        ep_clean = ica_s.apply(sub_raw, verbose=False)
        ep_clean.apply_baseline((0.0, 0.2), verbose=False)
        ep_clean.filter(
            0.5, 30.0, method="iir", iir_params=dict(order=4, ftype="butter"), verbose=False
        )
        s_x = drop_bad_trials(ep_clean.get_data(copy=True))
        n_tr, n_v = int(len(s_x) * 0.64), int(len(s_x) * 0.16)
        s_tr.append(s_x[:n_tr])
        s_v.append(s_x[n_tr : n_tr + n_v])
        s_te.append(s_x[n_tr + n_v :])

    n_tr_min, n_v_min, n_te_min = min(len(x) for x in s_tr), min(len(x) for x in s_v), min(len(x) for x in s_te)
    x_tr = np.concatenate([x[:n_tr_min] for x in s_tr], axis=0)
    y_tr = np.concatenate([np.full(n_tr_min, i) for i in range(len(WORDS))])
    x_v = np.concatenate([x[:n_v_min] for x in s_v], axis=0)
    y_v = np.concatenate([np.full(n_v_min, i) for i in range(len(WORDS))])
    x_te = np.concatenate([x[:n_te_min] for x in s_te], axis=0)
    y_te = np.concatenate([np.full(n_te_min, i) for i in range(len(WORDS))])

    x_tr = (x_tr - x_tr.mean(axis=-1, keepdims=True)) / (x_tr.std(axis=-1, keepdims=True) + 1e-8)
    x_v = (x_v - x_v.mean(axis=-1, keepdims=True)) / (x_v.std(axis=-1, keepdims=True) + 1e-8)
    x_te = (x_te - x_te.mean(axis=-1, keepdims=True)) / (x_te.std(axis=-1, keepdims=True) + 1e-8)
    subject_data_clean[s] = (x_tr, y_tr, x_v, y_v, x_te, y_te)

clean_accuracies = train_and_evaluate_eegnet(
    subject_data_clean,
    n_classes=len(WORDS),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    device=device,
)

print(f"=== Within-Subject 4-Class EEGNet Post-ICA Cleaned ({MODALITY.upper()}, 0.5-30 Hz) ===")
for s in SUBJECT_IDS:
    print(
        f"Subject {s:02d} -> Cleaned Accuracy: {clean_accuracies[s]:.2f}% "
        f"(Before ICA: {subject_accuracies[s]:.2f}%)"
    )

mean_orig = float(np.mean(list(subject_accuracies.values())))
mean_clean = float(np.mean(list(clean_accuracies.values())))
print("=" * 60)
print(f"Mean Pre-ICA Accuracy  : {mean_orig:.2f}%")
print(f"Mean Post-ICA Accuracy : {mean_clean:.2f}% (Delta: {mean_clean - mean_orig:+.2f}%)")
print("=" * 60)